# S09: calibrated HAR, a reverting-vs-shifted switching filter, and overnight-gap regimes

Pre-registration: [`S09_README.md`](S09_README.md). Engine: `src/equity_trading/s09.py` on the S08 paths.

**Fit on 2020–2022, evaluate on 2023–2025.** Every target is the forward β-hedged return (no α), as in S08 amendment A1.

- A. Calibrated log-HAR for next-session residual variance → V4 = s0 / λ_HAR.
- B. Two-state switching model of the s path (R: reverting, S: shifted), filtered P(R) from the second bar on.
- C. Overnight-gap regime: faded forward return after the first bar, by standardized gap size.

In [ ]:
TEST_START = "2023-01-01"
SAMPLE_EVERY = 4
EM_SEQUENCES = 60_000          # training stock-sessions sampled for the EM fit
ROUND_TRIP_BPS = 13.0

import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, s04, s05, s08, s09
from equity_trading.alpaca import ALPACA_VERSION, download_years

SYNTHETIC = os.environ.get("S09_SYNTHETIC") == "1"
YEARS = list(range(2020, 2026))                      # 2026 not used
CFG = s08.S08Config()
OUT = ROOT / "results" / ("s09_synthetic" if SYNTHETIC else "s09")
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
if SYNTHETIC:
    from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01
    ETF_OF = {s: SECTOR_ETF_01[k] for k, v in UNIVERSE_01.items() for s in v}
    YEARS, TEST_START, MIN_N, EM_SEQUENCES = [2024, 2025], "2025-01-01", 20, 5000
else:
    constituents = pd.read_csv(ROOT / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
    MIN_N = 50
STOCKS = sorted(ETF_OF)
ETFS = sorted(set(ETF_OF.values()))

## 1. Data and S08 paths

In [ ]:
t0 = time.time()
if SYNTHETIC:
    panel, _ = s04.synthetic_ohlc_panel(YEARS)
else:
    download_years(STOCKS + ETFS + ["SPY"], "15Min", YEARS, workers=8)
    panel = s04.cached_panel(STOCKS + ETFS + ["SPY"], "15Min", YEARS, ffill_bars=2)
have = set(panel.symbols)
TRADED = [s for s in STOCKS if s in have and ETF_OF[s] in have]
SC = np.array([panel.symbols.index(s) for s in TRADED])
EC = np.array([panel.symbols.index(ETF_OF[s]) for s in TRADED])
R = s08.compute(panel, SC, EC, CFG, verbose=False)
days = pd.to_datetime(panel.days)
DS = panel.day_start
D = panel.n_days
first_live = CFG.window_sessions
train_days = np.where((np.arange(D) >= first_live + 20) & (days < pd.Timestamp(TEST_START)))[0]
test_days = np.where(days >= pd.Timestamp(TEST_START))[0]
H = {"1d": 26, "5d": 130}
FWD = {h: s08.forward_sum(R["hedged"], n) for h, n in H.items()}
row_day = np.searchsorted(DS, np.arange(len(panel.index)), side="right") - 1
ALL_ROWS = np.arange(DS[first_live], len(panel.index) - max(H.values()) - 1, SAMPLE_EVERY)
TEST_ROWS = ALL_ROWS[np.isin(row_day[ALL_ROWS], test_days)]
print(f"paths in {(time.time() - t0) / 60:.1f} min | train days {len(train_days)} | test days {len(test_days)} | test cross-sections {len(TEST_ROWS):,}")

def mean_ic(sig, h, rows):
    ic = s08.rank_ic(sig, FWD[h], rows, min_n=MIN_N)
    d = pd.Series(ic, index=row_day[rows]).groupby(level=0).mean().dropna()
    import statsmodels.api as sm
    nw = sm.OLS(d.to_numpy(), np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": 1 if h == "1d" else 5})
    return float(d.mean()), float(nw.tvalues[0])

## 2. A — calibrated HAR (fit on training days, evaluated on test days)

In [ ]:
RV = s09.daily_rv(R["eps"], DS)
F = s09.har_features(RV)
HAR = s09.fit_har(F, train_days)
print("HAR coefficients [c, b1, b5, b20]:", np.round(HAR["coef"], 4), "| smearing", round(HAR["smear"], 4),
      "| training R2", round(HAR["r2"], 4), "| n", HAR["n"])
LAM = s09.har_lambda(F, HAR)
S4 = s09.apply_daily(R["s0"], LAM, DS)
har_rows = []
for name, sig in (("V0 S05", R["s0"]), ("V3 S08 equal-weight vol", R["s3"]), ("V4 calibrated HAR", S4)):
    for h in H:
        m, t = mean_ic(-sig, h, TEST_ROWS)
        har_rows.append({"variant": name, "horizon": h, "IC_test": m, "t_NW": t})
har_tab = pd.DataFrame(har_rows).pivot(index="variant", columns="horizon")
display(har_tab.round(4))

## 3. B — switching filter: fit on training sessions, filter every session

In [ ]:
rng = np.random.default_rng(42)
ylag, dy = s09.session_sequences(R["s0"], DS, train_days)
pick = rng.choice(len(ylag), size=min(EM_SEQUENCES, len(ylag)), replace=False)
t0 = time.time()
MS = s09.fit_switching(ylag[pick], dy[pick])
print(f"EM in {time.time() - t0:.0f}s, {MS['iterations']} iterations")
display(pd.Series({"k (pull per bar, state R)": MS["k"], "half-life in R (bars)": MS["half_life_bars_R"], "sigma": MS["sd"],
                   "P(R->R)": MS["P"][0, 0], "P(S->S)": MS["P"][1, 1], "stationary P(R)": MS["stationary"][0]}).round(4).to_frame())
t0 = time.time()
PR = s09.filter_prob_R(R["s0"], DS, MS)
print(f"filtered P(R) in {time.time() - t0:.0f}s")

In [ ]:
cond = {h: s09.conditional_table(R["s0"], PR, FWD[h], TEST_ROWS) for h in H}
display(pd.concat(cond, names=["horizon"]).round(2))
counts = s09.qualifying_counts(R["s0"], PR, TEST_ROWS)
all2 = (np.abs(R["s0"][TEST_ROWS]) >= 2).sum(1)
print(f"names per cross-section with |s|>=2: median {np.median(all2):.0f}; with |s|>=2 and P(R)>0.7: median {np.median(counts):.0f}, "
      f"10th pct {np.percentile(counts, 10):.0f}")
sig_w = -R["s0"] * PR
ms_rows = []
for name, sig in (("-s0", -R["s0"]), ("-s0 x P(R)", sig_w)):
    for h in H:
        m, t = mean_ic(sig, h, TEST_ROWS)
        ms_rows.append({"signal": name, "horizon": h, "IC_test": m, "t_NW": t})
ms_ic = pd.DataFrame(ms_rows).pivot(index="signal", columns="horizon")
display(ms_ic.round(4))
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for i, h in enumerate(H):
    c = cond[h]
    ax[i].errorbar(range(len(c)), c["contrarian_bps"], yerr=1.96 * c["se_bps"], marker="o", capsize=3, color="#2a78d6")
    ax[i].axhline(ROUND_TRIP_BPS, color="k", ls=":", lw=0.8, label="round trip")
    ax[i].set_xticks(range(len(c))); ax[i].set_xticklabels([str(b) for b in c.index], fontsize=7)
    ax[i].set_title(f"|s| >= 2: contrarian hedged bps by P(R), {h} (test)"); ax[i].legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. C — overnight-gap regime: fade the first bar's residual gap?

In [ ]:
gap = {"train": s09.gap_table(R["eps"], RV, DS, FWD, train_days), "test": s09.gap_table(R["eps"], RV, DS, FWD, test_days)}
display(pd.concat(gap, names=["period"]).round(2))

## 5. Pre-stated readings and summary

In [ ]:
def ic_of(tab, row, h): return float(tab.loc[row, ("IC_test", h)])
c5 = cond["5d"]
hi = c5[[b.left >= 0.7 for b in c5.index]]
lo = c5[[b.right <= 0.3 for b in c5.index]]
hi_bps = float((hi["contrarian_bps"] * hi["n"]).sum() / hi["n"].sum()) if len(hi) else np.nan
lo_bps = float((lo["contrarian_bps"] * lo["n"]).sum() / lo["n"].sum()) if len(lo) else np.nan
g5 = gap["test"].xs("5d", level="h")
big = g5[[b.left >= 3 for b in g5.index]]
big_bps = float((big["fade_bps"] * big["n"]).sum() / big["n"].sum()) if len(big) else np.nan
readings = {
    "H1 calibrated HAR worth porting (V4 test IC >= 1.10 x V0 at 1d and 5d)":
        ic_of(har_tab, "V4 calibrated HAR", "1d") >= 1.10 * ic_of(har_tab, "V0 S05", "1d") and
        ic_of(har_tab, "V4 calibrated HAR", "5d") >= 1.10 * ic_of(har_tab, "V0 S05", "5d"),
    "M1 switching filter worth a backtest (|s|>=2 & P(R)>0.7: 5d >= 25 bps, >= 10 bps above P(R)<=0.3, median names >= 25)":
        bool(hi_bps >= 25 and hi_bps - lo_bps >= 10 and np.median(counts) >= 25),
    "G1 gap fade worth a backtest (|g| >= 3: 5d faded return >= 25 bps, test)": bool(big_bps >= 25),
    "G2 gap continuation worth a backtest (|g| >= 3: 5d faded return <= -13 bps, test)": bool(big_bps <= -13),
}
display(pd.DataFrame({"value": {"P(R)>0.7 5d bps": hi_bps, "P(R)<=0.3 5d bps": lo_bps, "median qualifying names": float(np.median(counts)),
                                "|g|>=3 5d fade bps (test)": big_bps}}).round(2))
display(pd.Series(readings).to_frame("reading"))
summary = {"study": "S09", "synthetic": SYNTHETIC, "years": YEARS, "test_start": TEST_START, "har": {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in HAR.items()},
           "har_ic": {f"{a}|{b}": v for (a, b), v in har_tab.stack(level=1, future_stack=True)["IC_test"].items()},
           "switching": {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in MS.items()},
           "switching_ic": {f"{a}|{b}": v for (a, b), v in ms_ic.stack(level=1, future_stack=True)["IC_test"].items()},
           "conditional_5d": {str(k): v for k, v in c5["contrarian_bps"].items()},
           "conditional_1d": {str(k): v for k, v in cond["1d"]["contrarian_bps"].items()},
           "qualifying_names_median": float(np.median(counts)),
           "gap_test": {f"{h}|{b}": float(v) for (h, b), v in gap["test"]["fade_bps"].items()},
           "gap_train": {f"{h}|{b}": float(v) for (h, b), v in gap["train"]["fade_bps"].items()},
           "readings": readings, "versions": {"package": __version__, "s08": s08.S08_VERSION, "s09": s09.S09_VERSION, "panel": s04.PANEL_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
har_tab.to_csv(OUT / "har_ic.csv"); ms_ic.to_csv(OUT / "switching_ic.csv")
pd.concat(cond, names=["horizon"]).to_csv(OUT / "switching_conditional.csv"); pd.concat(gap, names=["period"]).to_csv(OUT / "gap_regime.csv")
print(json.dumps({k: bool(v) for k, v in readings.items()}, indent=2))